<a href="https://colab.research.google.com/github/Prem-Kumar2278/Transformers_mini_projects/blob/main/Transformers_mini_projects_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##  Translate in 4 lines

In [ ]:
!pip install -q transformers sentencepiece sacremoses

In [ ]:
from transformers import logging
logging.set_verbosity_error()   # hides extra warnings

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = "Helsinki-NLP/opus-mt-en-ur"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [ ]:
def translate(text):
    inputs = tokenizer(text, return_tensors="pt") #pythorch tenser pt    tensarflow works best on images and pytorch work best in resarche
    output = model.generate(**inputs, num_beams=4, max_new_tokens=100)
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(translate("The ball hit the window and it broke."))

گیند نے کھڑکی کو توڑ کر توڑ دیا ۔


In [ ]:
sentences = [
    "The ball hit the window and it broke.",
    "I love cricket.",
    "The weather is very hot today.",
    "Transformers changed artificial intelligence.",
]

for s in sentences:
    print("English:", s)
    print("Urdu   :", translate(s))
    print()

English: The ball hit the window and it broke.
german   : گیند نے کھڑکی کو توڑ کر توڑ دیا ۔

English: I love cricket.
german   : مجھے ڈرنے والوں سے محبت.

English: The weather is very hot today.
german   : آجکل موسم بہت گرم ہے ۔

English: Transformers changed artificial intelligence.
german   : ڈرنے والوں نے مصنوعی ذہانت تبدیل کر لی.



##Fill in the blank with BERT

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import pipeline
fill = pipeline("fill-mask", model="bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
sentence = "The cricket [MASK] was very fast."
results = fill(sentence, top_k=5)

print("Best sentence:", results[0]["sequence"])
print()

for r in results:
    print(f"{r['score'] * 100:5.1f}%  {r['sequence']}")

Best sentence: the cricket team was very fast.

 24.8%  the cricket team was very fast.
 22.6%  the cricket game was very fast.
 10.8%  the cricket pitch was very fast.
  6.6%  the cricket match was very fast.
  5.2%  the cricket season was very fast.


In [ ]:
sentences = [
    "I went to the [MASK] to buy medicine.",
    "I went to the [MASK] to watch a movie.",
]

for s in sentences:
    print(s)
    for r in fill(s, top_k=3):
        print(f"   {r['token_str']:<12} {r['score'] * 100:.1f}%")
    print()

I went to the [MASK] to buy medicine.
   store        68.4%
   pharmacy     11.6%
   market       9.0%

I went to the [MASK] to watch a movie.
   bathroom     13.0%
   theater      13.0%
   cinema       7.8%



In [ ]:
my_sentence = "Karachi is a very [MASK] city."   # change this sentence
for r in fill(my_sentence, top_k=5):
    print(f"{r['token_str']:<12} {r['score'] * 100:.1f}%")

cosmopolitan 13.2%
modern       7.0%
old          6.7%
busy         6.5%
large        4.8%


##Sentiment analysis

In [ ]:
from transformers import pipeline

In [ ]:
classifier = pipeline("sentiment-analysis")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
print(classifier("This course was really helpful!"))

[{'label': 'POSITIVE', 'score': 0.9994206428527832}]


##Question answering

In [ ]:
!pip install -q transformers

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

name = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForQuestionAnswering.from_pretrained(name)

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [ ]:
context = """Karachi is the largest city of Pakistan. It is located on the coast
of the Arabian Sea in the province of Sindh. The city is the main port and
the financial center of the country."""

In [ ]:
def ask(question):
    inputs = tokenizer(question, context, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs)

    start = torch.argmax(out.start_logits).item()   # best start word
    end = torch.argmax(out.end_logits).item()       # best end word

    if end < start:
        return "No clear answer found", 0.0

    answer = tokenizer.decode(inputs["input_ids"][0][start:end + 1], skip_special_tokens=True)

    start_p = torch.softmax(out.start_logits, dim=-1)[0, start].item()
    end_p = torch.softmax(out.end_logits, dim=-1)[0, end].item()
    return answer.strip(), start_p * end_p * 100

In [ ]:
questions = [
    "Which sea is Karachi located on?",
    "Which province is Karachi in?",
    "What is Karachi the main center of?",
]

for q in questions:
    answer, score = ask(q)
    print(f"Q: {q}")
    print(f"A: {answer}  ({score:.1f}%)")
    print()

Q: Which sea is Karachi located on?
A: Arabian Sea  (88.8%)

Q: Which province is Karachi in?
A: Sindh  (98.5%)

Q: What is Karachi the main center of?
A: financial center of the country  (75.2%)



##Summarization

In [ ]:
!pip install -q transformers sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import logging
logging.set_verbosity_error()   # hides extra warnings
name = "sshleifer/distilbart-cnn-12-6"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name)

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

In [ ]:
def summarize(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    output = model.generate(
        **inputs,
        num_beams=4,
        min_new_tokens=30,
        max_new_tokens=80,
        no_repeat_ngram_size=3,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
article = """
A school in Karachi has started a new programme to teach students how to use
artificial intelligence tools safely. The programme began this month and is open
to students from grade eight to grade ten. Every Saturday, students meet in the
computer lab, where teachers show them how language models work and how to check
their answers for mistakes. The school says many students already use AI tools
for homework, but few know how these tools make errors. Teachers hope the classes
will help students use AI as a helper and not as a shortcut. Parents were invited
to the first session, and many said they also wanted to learn. The school plans
to add a second class for teachers next term, and it may share its lessons with
other schools in the city if the programme is successful.
"""

print(summarize(article))

 A school in Karachi has started a new programme to teach students how to use AI tools safely . The programme began this month and is open to students from grade eight to grade ten .


In [ ]:
summary = summarize(article)
sentences = summary.split(". ")
print(". ".join(sentences[:3]))

 A school in Karachi has started a new programme to teach students how to use AI tools safely . The programme began this month and is open to students from grade eight to grade ten .


In [ ]:
for i in sentences:
    print(i)

 A school in Karachi has started a new programme to teach students how to use AI tools safely 
The programme began this month and is open to students from grade eight to grade ten .


##Chatbot over your own notes

In [ ]:
!pip install -q transformers pypdf

In [ ]:
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()                  # choose your PDF
pdf_name = list(uploaded.keys())[0]

reader = PdfReader(pdf_name)
text = ""
for page in reader.pages:
    text += (page.extract_text() or "") + "\n"

print(len(text.split()), "words read")

In [ ]:
def split_text(text, size=150, overlap=30):
    words = text.split()
    chunks = []
    step = size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + size])
        if chunk:
            chunks.append(chunk)
    return chunks

chunks = split_text(text)
print(len(chunks), "chunks")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

emb_name = "sentence-transformers/all-MiniLM-L6-v2"
emb_tok = AutoTokenizer.from_pretrained(emb_name)
emb_model = AutoModel.from_pretrained(emb_name)

In [ ]:
def embed(texts):
    inputs = emb_tok(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
    with torch.no_grad():
        out = emb_model(**inputs)
    mask = inputs["attention_mask"].unsqueeze(-1)
    vectors = (out.last_hidden_state * mask).sum(1) / mask.sum(1)   # average of the word vectors
    return torch.nn.functional.normalize(vectors, dim=1)

chunk_vectors = torch.cat([embed(chunks[i:i + 32]) for i in range(0, len(chunks), 32)])
print(chunk_vectors.shape)

In [ ]:
def find_chunks(question, k=3):
    q = embed([question])
    scores = (chunk_vectors @ q.T).squeeze(1)      # similarity of each chunk to the question
    top = torch.topk(scores, min(k, len(chunks)))
    return [(chunks[i], s.item()) for s, i in zip(top.values, top.indices)]

In [ ]:
from transformers import AutoModelForQuestionAnswering

qa_name = "distilbert-base-cased-distilled-squad"
qa_tok = AutoTokenizer.from_pretrained(qa_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_name)

In [ ]:
def answer_from(question, context):
    inputs = qa_tok(question, context, return_tensors="pt", truncation="only_second", max_length=384)
    with torch.no_grad():
        out = qa_model(**inputs)

    # allow answers only inside the chunk, not inside the question
    seq = inputs.sequence_ids(0)
    not_context = torch.tensor([s != 1 for s in seq])
    start_logits = out.start_logits.clone()
    end_logits = out.end_logits.clone()
    start_logits[0, not_context] = -1e4
    end_logits[0, not_context] = -1e4

    start = torch.argmax(start_logits).item()
    end = torch.argmax(end_logits).item()
    if end < start:
        return "", 0.0

    answer = qa_tok.decode(inputs["input_ids"][0][start:end + 1], skip_special_tokens=True)
    start_p = torch.softmax(start_logits, dim=-1)[0, start].item()
    end_p = torch.softmax(end_logits, dim=-1)[0, end].item()
    return answer.strip(), start_p * end_p * 100

In [ ]:
def ask(question):
    best = ("", 0.0, "")
    for chunk, similarity in find_chunks(question, k=3):
        answer, score = answer_from(question, chunk)
        if score > best[1]:
            best = (answer, score, chunk)
    return best

In [ ]:
while True:
    q = input("Ask a question (type quit to stop): ")
    if q.strip().lower() == "quit":
        break
    answer, score, chunk = ask(q)
    print("Answer:", answer, f"({score:.1f}%)")
    print("Found in:", chunk[:250], "...")
    print()

In [ ]:
answer, score, chunk = ask("How many layers does the encoder have?")
print(answer, f"({score:.1f}%)")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = "Helsinki-NLP/opus-mt-en-ur"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name)

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/816k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/848k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  306MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:
def translate(text):
    inputs = tokenizer(text, return_tensors="pt")
    output = model.generate(**inputs, num_beams=4, max_new_tokens=100)
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(translate("The ball hit the window and it broke."))

model.safetensors: reconstructing file:   0%|          |  0.00B /  306MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


گیند نے کھڑکی کو توڑ کر توڑ دیا ۔


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, logging

logging.set_verbosity_error()   # hides extra warnings

name = "Helsinki-NLP/opus-mt-en-ur"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name)

def translate(text):
    inputs = tokenizer(text, return_tensors="pt")
    output = model.generate(**inputs, num_beams=4)
    return tokenizer.decode(output[0], skip_special_tokens=True)

sentences = [
    "The ball hit the window and it broke.",
    "I love cricket.",
    "The weather is very hot today.",
    "Transformers changed artificial intelligence.",
]

for s in sentences:
    print("English:", s)
    print("Urdu   :", translate(s))
    print()

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

English: The ball hit the window and it broke.
Urdu   : گیند نے کھڑکی کو توڑ کر توڑ دیا ۔

English: I love cricket.
Urdu   : مجھے ڈرنے والوں سے محبت.

English: The weather is very hot today.
Urdu   : آجکل موسم بہت گرم ہے ۔

English: Transformers changed artificial intelligence.
Urdu   : ڈرنے والوں نے مصنوعی ذہانت تبدیل کر لی.



In [ ]:
from transformers import pipeline

fill = pipeline("fill-mask", model="distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
results = fill("The ball hit the [MASK].", top_k=5)

for r in results:
    print(f"{r['token_str']:<10} {r['score'] * 100:.1f}%")

ground     6.6%
basket     3.3%
ball       2.9%
pitcher    2.4%
batter     2.2%


In [ ]:
results = fill("The ball hit the [MASK].", top_k=5)

best = results[0]
print(best["sequence"])
print(f"Word chosen: {best['token_str']} ({best['score'] * 100:.1f}%)")

the ball hit the ground.
Word chosen: ground (6.6%)


In [ ]:
from transformers import pipeline

In [ ]:
classifier = pipeline("sentiment-analysis")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
print(classifier("This course was really helpful!"))

[{'label': 'POSITIVE', 'score': 0.9994206428527832}]


##Build a Text Generator in Python with Hugging Face second
Date 2/11/2026

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import logging
logging.set_verbosity_error()   # hides extra warnings

name = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name)

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
# def generate_text(prompt, max_new_tokens=40, temperature=0.8, top_k=50):
#     inputs = tokenizer(prompt, return_tensors="pt")
#     output = model.generate(
#         **inputs,
#         do_sample=True,
#         temperature=temperature,
#         top_k=top_k,
#         max_new_tokens=max_new_tokens,
#         pad_token_id=tokenizer.eos_token_id,
#     )
#     return tokenizer.decode(output[0], skip_special_tokens=True)


# # Interactive loop for dynamic user input
# if __name__ == "__main__":
#     print("Text Generation Assistant (Type 'exit' or 'quit' to stop)\n")

#     while True:
#         # Get user prompt from input
#         user_prompt = input("Enter your prompt: ").strip()

#         # Allow the user to exit the loop easily
#         if user_prompt.lower() in ["exit", "quit"]:
#             print("Exiting...")
#             break

#         if not user_prompt:
#             print("Please enter a valid prompt.\n")
#             continue

#         print("\nGenerating...")
#         generated_result = generate_text(user_prompt)
#         print(f"\nResult:\n{generated_result}\n" + "-"*50 + "\n")






def generate_text(prompt, max_new_tokens=40, temperature=0.8, top_k=50):
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(
        **inputs,
        do_sample=True,
        temperature=temperature,
        top_k=top_k,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(generate_text("Karachi is a city where"))

Karachi is a city where people of all ages, social classes and walks of life come together to enjoy a great city life.

A visit to the city can give you an insight into the life of the people of the


In [ ]:
prompts = [
    "Karachi is a city where",
    "The best way to learn programming is",
    "Once upon a time, a small boy found",
]

for p in prompts:
    print("Prompt:", p)
    print("Text  :", generate_text(p))
    print()

Prompt: Karachi is a city where
Text  : Karachi is a city where the Indian Independence and Islamic Revolution has taken place in the past 50 years. But the city is a cultural and cultural hub of the country. The city is a symbol of Indian culture and culture

Prompt: The best way to learn programming is
Text  : The best way to learn programming is through the use of the programming language itself. Most people don’t realize the difference between programming languages and computer algorithms, and they use programming languages interchangeably. To determine which is which, consider looking at

Prompt: Once upon a time, a small boy found
Text  : Once upon a time, a small boy found himself unexpectedly surrounded by people with different opinions and backgrounds. He met various individuals who shared his own life experiences, which made him feel more connected to them than ever before. One day, he invited them



In [ ]:
prompt = "The future of artificial intelligence is"

for t in [0.2, 0.8, 1.5]:
    print(f"Temperature {t}:")
    print(generate_text(prompt, temperature=t))
    print()

Temperature 0.2:
The future of artificial intelligence is a very exciting one.

Artificial intelligence is a field of study that is constantly evolving and developing. It is a field that is constantly changing and evolving. It is a field that is constantly changing

Temperature 0.8:
The future of artificial intelligence is largely unknown, although one researcher in Cambridge, Massachusetts, says that “the prospect of robots in the workplace” is imminent.

What is AI?

The term artificial intelligence stands for the

Temperature 1.5:
The future of artificial intelligence is all about the way we use it. Many users have argued, however naively, that today, a computer’s ability to mimic human intelligence goes against the common wisdom the fact that humans tend to



In [ ]:
inputs = tokenizer("The future of artificial intelligence is",
                   return_tensors="pt")

output = model.generate(**inputs, do_sample=False, max_new_tokens=40,
                        pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(output[0], skip_special_tokens=True))

The future of artificial intelligence is a question that is still being debated.

The future of artificial intelligence is a question that is still being debated.

Artificial intelligence is a field of study that is still in its infancy.


## Zero-Short Classification

In [ ]:
!pip install -q transformers

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import logging
logging.set_verbosity_error()   # hides extra warnings

In [ ]:
name = "typeform/distilbert-base-uncased-mnli"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
print(model.config.id2label)

{0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}


In [ ]:
entail_id = [i for i, l in model.config.id2label.items() if l.lower().startswith("entail")][0]
print(entail_id)

In [ ]:
def classify(text, labels):
    scores = []
    for label in labels:
        hypothesis = f"This text is about {label}."
        inputs = tokenizer(text, hypothesis, return_tensors="pt", truncation=True)
        with torch.no_grad():
            out = model(**inputs)
        scores.append(out.logits[0, entail_id].item())

    percents = torch.softmax(torch.tensor(scores), dim=0) * 100
    results = sorted(zip(labels, percents.tolist()), key=lambda x: x[1], reverse=True)
    return results

In [ ]:
text = "The government announced a new tax on petrol today."
labels = ["politics", "sports", "technology", "health"]

for label, score in classify(text, labels):
    print(f"{label:<12} {score:.1f}%")

#Name Finder

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import pipeline
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
text = "Ali studies at the University of Sindh in Jamshoro, and he met a Google engineer in Karachi."

for e in ner(text):
    print(f"{e['word']:<25} {e['entity_group']:<6} {e['score'] * 100:.1f}%")

Ali                       PER    99.9%
University of Sindh       ORG    99.8%
Jamshoro                  LOC    99.6%
Google                    ORG    99.2%
Karachi                   LOC    99.9%


In [ ]:
from transformers import pipeline
ner = pipeline("ner", model="dslim/bert-base-NER") # aggregation_strategy="simple")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
text = "Ali studies at the University of Sindh in Jamshoro, and he met a Google engineer in Karachi."


for e in ner(text):
  print(f"{e['word']:<25} {e['score'] * 100:.1f}%")
#     print(f"{e['word']:<25} {e['entity_group']:<6} {e['score'] * 100:.1f}%")
#

Ali                       99.9%
University                99.8%
of                        99.8%
Sindh                     99.8%
Jam                       99.8%
##sh                      99.5%
##oro                     99.6%
Google                    99.2%
Karachi                   99.9%


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

In [ ]:
name = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForTokenClassification.from_pretrained(name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
text = "Ali studies at the University of Sindh in Jamshoro, and he met a Google engineer in Karachi."

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    out = model(**inputs)

pred_ids = torch.argmax(out.logits, dim=-1)[0]
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, pid in zip(tokens, pred_ids):
    print(f"{token:<15} {model.config.id2label[pid.item()]}")

[CLS]           O
Ali             B-PER
studies         O
at              O
the             O
University      B-ORG
of              I-ORG
Sindh           I-ORG
in              O
Jam             B-LOC
##sh            I-LOC
##oro           I-LOC
,               O
and             O
he              O
met             O
a               O
Google          B-ORG
engineer        O
in              O
Karachi         B-LOC
.               O
[SEP]           O


In [ ]:
from transformers import pipeline

# Re-initialize the pipeline with aggregation_strategy to get 'entity_group'
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

sentences = [
    "Lionel Messi joined Inter Miami in the United States.",
    "Microsoft opened a new office in Lahore last year.",
    "The Indus River flows through Sindh.",
]

for s in sentences:
    print(s)
    for e in ner(s):
        print(f"{e['word']:<20} {e['entity_group']}")
    print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Lionel Messi joined Inter Miami in the United States.
Lionel Messi         PER
Inter Miami          ORG
United States        LOC

Microsoft opened a new office in Lahore last year.
Microsoft            ORG
Lahore               LOC

The Indus River flows through Sindh.
In                   LOC
##dus River          LOC
Sindh                LOC



In [ ]:
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="first")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
good = [e for e in ner(text) if e["score"] > 0.90]
good

[{'entity_group': 'PER',
  'score': np.float32(0.9994272),
  'word': 'Ali',
  'start': 0,
  'end': 3},
 {'entity_group': 'ORG',
  'score': np.float32(0.99820596),
  'word': 'University of Sindh',
  'start': 19,
  'end': 38},
 {'entity_group': 'LOC',
  'score': np.float32(0.99826837),
  'word': 'Jamshoro',
  'start': 42,
  'end': 50},
 {'entity_group': 'ORG',
  'score': np.float32(0.9923517),
  'word': 'Google',
  'start': 65,
  'end': 71},
 {'entity_group': 'LOC',
  'score': np.float32(0.99899894),
  'word': 'Karachi',
  'start': 84,
  'end': 91}]

## Fine-tune your own model

In [ ]:
!pip install -q transformers seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
)
from transformers import logging
logging.set_verbosity_error()   # hides extra warnings

In [ ]:
data = [
    (["I","ate","biryani","in","Karachi","."],                    ["O","O","B-FOOD","O","B-CITY","O"]),
    (["She","loves","chicken","karahi","in","Lahore","."],        ["O","O","B-FOOD","I-FOOD","O","B-CITY","O"]),
    (["We","had","fried","rice","in","Islamabad","."],            ["O","O","B-FOOD","I-FOOD","O","B-CITY","O"]),
    (["He","cooked","daal","for","dinner","."],                   ["O","O","B-FOOD","O","O","O"]),
    (["They","travelled","to","Hyderabad","for","a","wedding","."],["O","O","O","B-CITY","O","O","O","O"]),
    (["My","mother","makes","great","nihari","."],                ["O","O","O","O","B-FOOD","O"]),
    (["The","bus","to","Multan","leaves","at","noon","."],        ["O","O","O","B-CITY","O","O","O","O"]),
    (["Ali","bought","samosa","in","Quetta","."],                 ["O","O","B-FOOD","O","B-CITY","O"]),
    (["We","ordered","butter","chicken","and","naan","in","Peshawar","."],
                                                                   ["O","O","B-FOOD","I-FOOD","O","B-FOOD","O","B-CITY","O"]),
    (["Sara","visited","Karachi","last","year","."],              ["O","O","B-CITY","O","O","O"]),
    (["Haleem","is","popular","in","Sukkur","."],                 ["B-FOOD","O","O","O","B-CITY","O"]),
    (["He","dislikes","fried","fish","."],                        ["O","O","B-FOOD","I-FOOD","O"]),
    (["The","train","reached","Rawalpindi","early","."],          ["O","O","O","B-CITY","O","O"]),
    (["She","made","pasta","at","home","."],                      ["O","O","B-FOOD","O","O","O"]),
    (["We","ate","kebab","near","Faisalabad","."],                ["O","O","B-FOOD","O","B-CITY","O"]),
    (["Dinner","in","Karachi","was","biryani","."],               ["O","O","B-CITY","O","B-FOOD","O"]),
]

In [ ]:
len(data)

16

In [ ]:
train_data = data[:12]   # used to train
test_data  = data[12:]   # kept aside to check the model

In [ ]:
data[:12]

[(['I', 'ate', 'biryani', 'in', 'Karachi', '.'],
  ['O', 'O', 'B-FOOD', 'O', 'B-CITY', 'O']),
 (['She', 'loves', 'chicken', 'karahi', 'in', 'Lahore', '.'],
  ['O', 'O', 'B-FOOD', 'I-FOOD', 'O', 'B-CITY', 'O']),
 (['We', 'had', 'fried', 'rice', 'in', 'Islamabad', '.'],
  ['O', 'O', 'B-FOOD', 'I-FOOD', 'O', 'B-CITY', 'O']),
 (['He', 'cooked', 'daal', 'for', 'dinner', '.'],
  ['O', 'O', 'B-FOOD', 'O', 'O', 'O']),
 (['They', 'travelled', 'to', 'Hyderabad', 'for', 'a', 'wedding', '.'],
  ['O', 'O', 'O', 'B-CITY', 'O', 'O', 'O', 'O']),
 (['My', 'mother', 'makes', 'great', 'nihari', '.'],
  ['O', 'O', 'O', 'O', 'B-FOOD', 'O']),
 (['The', 'bus', 'to', 'Multan', 'leaves', 'at', 'noon', '.'],
  ['O', 'O', 'O', 'B-CITY', 'O', 'O', 'O', 'O']),
 (['Ali', 'bought', 'samosa', 'in', 'Quetta', '.'],
  ['O', 'O', 'B-FOOD', 'O', 'B-CITY', 'O']),
 (['We', 'ordered', 'butter', 'chicken', 'and', 'naan', 'in', 'Peshawar', '.'],
  ['O', 'O', 'B-FOOD', 'I-FOOD', 'O', 'B-FOOD', 'O', 'B-CITY', 'O']),
 (['Sara', 

In [ ]:
data[12:]

[(['The', 'train', 'reached', 'Rawalpindi', 'early', '.'],
  ['O', 'O', 'O', 'B-CITY', 'O', 'O']),
 (['She', 'made', 'pasta', 'at', 'home', '.'],
  ['O', 'O', 'B-FOOD', 'O', 'O', 'O']),
 (['We', 'ate', 'kebab', 'near', 'Faisalabad', '.'],
  ['O', 'O', 'B-FOOD', 'O', 'B-CITY', 'O']),
 (['Dinner', 'in', 'Karachi', 'was', 'biryani', '.'],
  ['O', 'O', 'B-CITY', 'O', 'B-FOOD', 'O'])]

In [ ]:
labels_list = ["O", "B-FOOD", "I-FOOD", "B-CITY", "I-CITY"]
label2id = {label: i for i, label in enumerate(labels_list)} #enumerater convert evry word  into numbers

print(label2id)

{'O': 0, 'B-FOOD': 1, 'I-FOOD': 2, 'B-CITY': 3, 'I-CITY': 4}


In [ ]:
id2label = {i: label for label, i in label2id.items()}
print(id2label)

{0: 'O', 1: 'B-FOOD', 2: 'I-FOOD', 3: 'B-CITY', 4: 'I-CITY'}


In [ ]:
name = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(name)

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
words = ["Haleem", "is", "popular", "in", "Sukkur", "."]
enc = tokenizer(words, is_split_into_words=True)

print(tokenizer.convert_ids_to_tokens(enc["input_ids"]))
print(enc.word_ids())

['[CLS]', 'Hale', '##em', 'is', 'popular', 'in', 'Su', '##k', '##ku', '##r', '.', '[SEP]']
[None, 0, 0, 1, 2, 3, 4, 4, 4, 4, 5, None]
